In [1]:
pip install pandas-datareader yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 7.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pandas-datareader]curl_cffi]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# montly time window from January 2000 to the present to capture multiple 
# monetary policy tighteming and easing cycles

In [4]:
import datetime
import numpy as np 
import pandas as pd 
import pandas_datareader.data as web

In [5]:
start_date = datetime.datetime(2000, 1, 1)
end_date = datetime.datetime.now()

# define observation time horizon

In [6]:
fred_tickers = {
    'FEDFUNDS' : 'policy_rate', # federal funds rate
    'CPIAUCSL' : 'cpi', # CPI for all urban consumers 
    'UNRATE' : 'unemployment_rate', # civilian unemployment rate
}

In [7]:
df_macro = web.DataReader(list(fred_tickers.keys()), 'fred', start_date, end_date)
df_macro = df_macro.rename(columns = fred_tickers)

# fetch empirical time-series data directly from FRED servers

In [8]:
# the price level CPI must be converted into a Year-over-Year percentage change to compute
# the headline inflation rate

In [9]:
df_macro['inflation_rate_yoy'] = df_macro['cpi'].pct_change(12)*100

# compute yoy inflation rate (12-month percentage change)

/var/folders/s4/rcr39x8d50j2rvsrqz_8j0sm0000gn/T/ipykernel_17817/117282576.py:1: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_macro['inflation_rate_yoy'] = df_macro['cpi'].pct_change(12)*100


In [10]:
df_macro['rate_change_1m'] = df_macro['policy_rate'].diff(1)

# compute 1 month change in policy interest rate

In [11]:
df_macro['tightening_regime'] = np.where(df_macro['rate_change_1m'] > 0, 1, 0)

# create a monetary tightening regime indicator (1 if rate hike, 0 otherwise)

In [12]:
print('--- [Engineered Features] First 5 Available Observations ---')
# Drop initial 12 NaN rows resulting from the 12-month lag transformation
df_clean_macro = df_macro.dropna()
df_clean_macro.head()

--- [Engineered Features] First 5 Available Observations ---


,policy_rate,cpi,unemployment_rate,inflation_rate_yoy,rate_change_1m,tightening_regime
DATE,,,,,,
2001-01-01,5.98,175.6,4.2,3.721205,-0.42,0
2001-02-01,5.49,176.0,4.2,3.529412,-0.49,0
2001-03-01,5.31,176.1,4.3,2.982456,-0.18,0
2001-04-01,4.80,176.4,4.4,3.218256,-0.51,0
2001-05-01,4.21,177.3,4.3,3.563084,-0.59,0
